# _n_-Gram Analysis

Collocations are often a sensible concept to account for in a language model, but that is all the more true when dealing with alliterative poetry, where poetic formulae have a significant place in the discourse about the mode of composition and the role of borrowing between poems. To demonstrate one possible way into the subject, we'll distill _n_-grams from ASPR, the Anglo-Saxon Poetic Records as distributed in XML and plaintext formats (among others) at the [Oxford Text Archive](https://ota.bodleian.ox.ac.uk/repository/xmlui/handle/20.500.12024/3009). This script assumes that distribution has been unpacked into a folder `../corpora/aspr/`.

Please note that the CLTK code below assumes you have installed CLTK 1.5 (`pip install cltk==1.5.0`), which requires Python 3.9, so if you wish to run this code or use that version of CLTK, make sure to set up a Python environment accordingly.

In [1]:
import json
from pathlib import Path
from collections import Counter
from lxml import etree
from nltk.collocations import BigramCollocationFinder,TrigramCollocationFinder
from nltk.metrics import BigramAssocMeasures,TrigramAssocMeasures
# The following functionality assumes CLTK 1.5, which only works on Python 3.7, 3.8, or 3.9!
from cltk.lemmatize.ang import OldEnglishDictionaryLemmatizer
lem = OldEnglishDictionaryLemmatizer()

In [2]:
xml_file = Path.cwd().parent / 'corpora' / 'aspr' / '3009.xml'

# Normalization matrix:
substitutions = {
    'ę': 'æ',
    'ð': 'þ',
    ',': '',
    '.': '',
    ':': '',
    ';': '',
    '?': '',
    '!': '',
    '-': '',
    '–': '',
    '"': '',
    "'": '',
    '[': '',
    ']': ''
}

# Token normalization:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

In [3]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus = dict()
tree = etree.parse(xml_file, parser=parser)
root = tree.getroot()
for poem in root.iter('{http://www.tei-c.org/ns/1.0}div'):
    verses = dict()
    title = poem.find('./{http://www.tei-c.org/ns/1.0}head').text
    line_no = 1
    for line in poem.iter('{http://www.tei-c.org/ns/1.0}l'):
        if len(line.findall('./{http://www.tei-c.org/ns/1.0}caesura')) > 0:
            caesura = line.find('./{http://www.tei-c.org/ns/1.0}caesura')
            caesura.text = '|||'
            tokens = []
            halflines = normalize(etree.tostring(line, method='text', encoding='unicode')).split('|||')
            verses[str(line_no) + 'a'] = halflines[0].split()
            verses[str(line_no) + 'b'] = halflines[1].split()
        else:
            halfline = normalize(etree.tostring(line, method='text', encoding='unicode'))
            verses[str(line_no) + 'a'] = halfline.split()
        line_no += 1
    corpus[title] = verses

_out_path = Path.cwd().parent / 'corpora' / 'aspr' / 'aspr.json'
with open(_out_path, 'w', encoding='utf-8') as outfile:
    json.dump(corpus, outfile, ensure_ascii=False, indent=4)




For best results, we should lemmatize our corpus. Unfortunately, the only lemmatizer available that does not rely on a (subscription to a) large language model is a superseded version of CLTK, which does rather a poor job. But at least it will lemmatize pronouns and forms of the verb for "to be", which helps a little.

In [4]:
corpus_lemmatized = dict()
for title,verses in corpus.items():
    poem = dict()
    for verse_id,tokens in verses.items():
        lemmatized_verse = []
        for token in tokens:
            lemma = lem(token)
            lemmatized_verse.append(lemma)
        poem[verse_id] = lemmatized_verse
    corpus_lemmatized[title] = poem

_out_path = Path.cwd().parent / 'corpora' / 'aspr' / 'aspr-lemmas.json'
with open(_out_path, 'w', encoding='utf-8') as outfile:
    json.dump(corpus_lemmatized, outfile, ensure_ascii=False, indent=4)


In [5]:
all_verses = [verse for verse in poem for poem in corpus.values() for verse in poem.values()]
all_tokens = [token for verse in all_verses for token in verse]
all_verses_lemmatized = [verse for verse in poem for poem in corpus_lemmatized.values() for verse in poem.values()]
all_tokens_lemmatized = [token for verse in all_verses_lemmatized for token in verse]

If we don't remove stopwords from our corpus, the top-ranking _n_-grams will all be "þæt he" and "ne mæg" and so on. CLTK ships with a stopword list, but with Old English I've found it best to remove the 300 or so most frequent words (always best to inspect a range of words on either side of the cutoff point):

In [6]:
frequencies = Counter(all_tokens)
stops = [k for k,v in frequencies.most_common(350)]
stopped_tokens = [token for token in all_tokens if token not in stops]

frequencies_lemmatized = Counter(all_tokens_lemmatized)
stops_lemmatized = [k for k,v in frequencies_lemmatized.most_common(350)]
stopped_tokens_lemmatized = [token for token in all_tokens_lemmatized if token not in stops_lemmatized]

In [7]:
bigrams_lemmatized = BigramCollocationFinder.from_words(stopped_tokens_lemmatized)
trigrams_lemmatized = TrigramCollocationFinder.from_words(stopped_tokens_lemmatized)

In [8]:
bigrams_lemmatized.nbest(BigramAssocMeasures.likelihood_ratio, 50)

[('agifan', 'ondsware'),
 ('ecnes', 'awa'),
 ('pater', 'noster'),
 ('sæm', 'tweonum'),
 ('wigendra', 'hleo'),
 ('sinc', 'brytta'),
 ('samod', 'ætgædere'),
 ('gewrit', 'secgaþ'),
 ('salomon', 'cuæþ'),
 ('beowulf', 'maþelian'),
 ('feondas', 'fæcne'),
 ('har', 'hilderinc'),
 ('heanne', 'beam'),
 ('geteled', 'rime'),
 ('yþ', 'gewealc'),
 ('torht', 'ontynan'),
 ('ent', 'geweorc'),
 ('nergend', 'usser'),
 ('wordhord', 'onleac'),
 ('maþelian', 'ecgþeowes'),
 ('aþ', 'swerian'),
 ('wegas', 'wise'),
 ('andreas', 'agifan'),
 ('petrus', 'paulus'),
 ('costunge', 'cleopedan'),
 ('feam', 'siþum'),
 ('wilde', 'deor'),
 ('ad', 'te'),
 ('suþan', 'norþan'),
 ('wlitig', 'wynsum'),
 ('crux', 'christi'),
 ('fæle', 'sceap'),
 ('magnam', 'misericordiam'),
 ('unrihtes', 'wyrceaþ'),
 ('geata', 'leode'),
 ('hroþgar', 'maþelian'),
 ('leofne', 'mannan'),
 ('nihtlangne', 'fyrst'),
 ('saturnus', 'forhwon'),
 ('norþan', 'eastan'),
 ('miserere', 'mei'),
 ('saluum', 'fac'),
 ('sodoman', 'gomorran'),
 ('idel', 'gylp'),


In [9]:
trigrams_lemmatized.nbest(TrigramAssocMeasures.likelihood_ratio, 50)

[('andreas', 'agifan', 'ondsware'),
 ('eadga', 'agifan', 'ondsware'),
 ('eadge', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'leofum'),
 ('agifan', 'ondsware', 'feondlice'),
 ('agifan', 'ondsware', 'gæstgehygd'),
 ('gieldum', 'agifan', 'ondsware'),
 ('himþaearmsceapen', 'agifan', 'ondsware'),
 ('æwitan', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'forhtafongen'),
 ('agifan', 'ondsware', 'geþreatast'),
 ('agifan', 'ondsware', 'þristlice'),
 ('agifan', 'ondsware', 'wiþsacan'),
 ('unforhte', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'iuliana'),
 ('agifan', 'ondsware', 'fyrhþ'),
 ('aglæca', 'agifan', 'ondsware'),
 ('yldra', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'beforan'),
 ('agifan', 'ondsware', 'geweorþan'),
 ('nabban', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'usic'),
 ('agifan', 'ondsware', 'oncnawan'),
 ('agifan', 'ondsware', 'gleaw'),
 ('cwen', 'agifan', 'ondsware'),
 ('agifan', 'ondsware', 'eaþe'),
 ('agifan', 'ondsware', 'fah'),
 ('beowulf', 'maþelian', '

According to these data, phrases for "answered" are especially frequent. We may try the nonlemmatized data as well, for comparison:

In [10]:
bigrams = BigramCollocationFinder.from_words(stopped_tokens)
trigrams = TrigramCollocationFinder.from_words(stopped_tokens)

In [ ]:
bigrams.nbest(BigramAssocMeasures.likelihood_ratio, 50)

[('ecnesse', 'awa'),
 ('saga', 'hatte'),
 ('pater', 'noster'),
 ('dæges', 'nihtes'),
 ('feowertig', 'daga'),
 ('sæm', 'tweonum'),
 ('ageaf', 'ondsware'),
 ('sinces', 'brytta'),
 ('bu', 'tu'),
 ('gewritu', 'secgaþ'),
 ('wigendra', 'hleo'),
 ('samod', 'ætgædere'),
 ('haliges', 'gastes'),
 ('wunden', 'gold'),
 ('beowulf', 'maþelode'),
 ('salomon', 'cuæþ'),
 ('uppe', 'englum'),
 ('torht', 'ontyned'),
 ('andreas', 'agef'),
 ('feondas', 'fæcne')]

In [ ]:
trigrams.nbest(TrigramAssocMeasures.likelihood_ratio, 50)

[('feowertig', 'daga', 'fæsten'),
 ('saga', 'hatte', 'wunderlicu'),
 ('beowulf', 'maþelode', 'ecgþeowes'),
 ('siþfæt', 'saga', 'hatte'),
 ('herige', 'ecnesse', 'awa'),
 ('ecnesse', 'awa', 'belgan'),
 ('ecnesse', 'awa', 'getimbrad'),
 ('ecnesse', 'awa', 'heahbliss'),
 ('ecnesse', 'awa', 'heahesta'),
 ('ecnesse', 'awa', 'æþelnes'),
 ('gebletsad', 'ecnesse', 'awa'),
 ('nigende', 'saga', 'hatte'),
 ('saga', 'hatte', 'byledbreost'),
 ('saga', 'hatte', 'scirenige'),
 ('searosæled', 'saga', 'hatte'),
 ('seolhbaþo', 'saga', 'hatte'),
 ('lærdest', 'ecnesse', 'awa'),
 ('ecnesse', 'awa', 'wunaþ'),
 ('meted', 'ecnesse', 'awa'),
 ('saga', 'hatte', 'hwa')]